# EC. Primer Parcial — Proyecto Práctico HPC
## Integrante 3 — Experimentación, Speedup y Eficiencia

**Rama:** `feature/experimentos`

---

## Objetivo de esta parte

En esta etapa no se cambia el algoritmo secuencial ni el paralelo.

El objetivo es tomar los tiempos ya obtenidos y convertirlos en información útil para comparar el rendimiento.

Se trabajará con:

- 3 mediciones de la versión secuencial;
- 3 mediciones con 1 worker;
- 3 mediciones con 2 workers;
- 3 mediciones con 4 workers;
- tiempo promedio;
- Speedup;
- eficiencia.

Los resultados obtenidos aquí podrán ser utilizados después por el integrante encargado de las gráficas y del análisis visual.


# 1. ¿Por qué repetir cada prueba?

Una sola medición puede verse afectada por otros procesos del sistema operativo, uso de memoria, temperatura del procesador o carga de la computadora.

Por eso se realizaron **3 ejecuciones por configuración**.

El promedio permite obtener una medida más estable:

\[
T_{promedio} = \frac{T_1 + T_2 + T_3}{3}
\]

> En esta etapa no se inventan tiempos: se leen directamente los CSV generados por las etapas anteriores.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Permite ejecutar el notebook desde la carpeta notebooks/
# o desde la raíz del proyecto.
cwd = Path.cwd()

if (cwd / "resultados").exists():
    RESULTADOS = cwd / "resultados"
elif (cwd.parent / "resultados").exists():
    RESULTADOS = cwd.parent / "resultados"
else:
    raise FileNotFoundError("No se encontró la carpeta resultados/")

print("Carpeta de resultados:", RESULTADOS.resolve())


# 2. Cargar los resultados existentes

La versión secuencial guardó sus mediciones en:

```text
resultados/secuencial.csv
```

La versión paralela guardó las mediciones con 1, 2 y 4 workers en:

```text
resultados/paralelo.csv
```

Primero verificaremos el contenido de ambos archivos.


In [ ]:
secuencial = pd.read_csv(RESULTADOS / "secuencial.csv")
paralelo = pd.read_csv(RESULTADOS / "paralelo.csv")

print("Resultados secuenciales:")
display(secuencial)

print("\nResultados paralelos:")
display(paralelo)


# 3. Calcular el promedio secuencial

El archivo secuencial ya contiene una fila de promedio, pero lo volveremos a calcular a partir de las tres pruebas reales para verificarlo.

La versión secuencial es útil como una comparación adicional para saber si el costo de crear procesos paralelos realmente compensa.


In [ ]:
secuencial_pruebas = secuencial[
    secuencial["prueba"].astype(str).str.lower() != "promedio"
].copy()

secuencial_pruebas["tiempo_segundos"] = pd.to_numeric(
    secuencial_pruebas["tiempo_segundos"]
)

tiempo_secuencial = secuencial_pruebas["tiempo_segundos"].mean()

print(f"Tiempo promedio secuencial: {tiempo_secuencial:.6f} s")


# 4. Promedio por número de workers

Ahora agrupamos las pruebas paralelas por la columna `workers`.

Para cada configuración calculamos:

\[
T_p = \frac{T_{p,1} + T_{p,2} + T_{p,3}}{3}
\]

donde `p` puede ser 1, 2 o 4 workers.


In [ ]:
promedios = (
    paralelo
    .groupby("workers", as_index=False)["tiempo_segundos"]
    .mean()
    .rename(columns={"tiempo_segundos": "tiempo_promedio_segundos"})
)

display(promedios)


# 5. Calcular Speedup

La consigna define:

\[
S_p = \frac{T_1}{T_p}
\]

Donde:

- \(T_1\) es el tiempo promedio con **1 worker**;
- \(T_p\) es el tiempo promedio con `p` workers.

Interpretación sencilla:

- `Speedup = 1`: no hubo aceleración respecto a 1 worker;
- `Speedup > 1`: hubo una mejora;
- `Speedup = 2`: la configuración fue aproximadamente dos veces más rápida que 1 worker.


In [ ]:
T1 = float(
    promedios.loc[
        promedios["workers"] == 1,
        "tiempo_promedio_segundos"
    ].iloc[0]
)

promedios["speedup"] = T1 / promedios["tiempo_promedio_segundos"]

display(promedios)


# 6. Calcular eficiencia

La eficiencia se calcula mediante:

\[
E_p = \frac{S_p}{p}
\]

Donde:

- \(S_p\) es el Speedup;
- \(p\) es el número de workers.

Una eficiencia menor al 100 % es normal, porque existen costos extra de creación de procesos, distribución de datos, comunicación y combinación de resultados.


In [ ]:
promedios["eficiencia"] = (
    promedios["speedup"] / promedios["workers"]
)

display(promedios)


# 7. Comparación adicional contra la versión secuencial

El Speedup oficial anterior usa 1 worker como referencia, tal como indica la consigna.

Sin embargo, también es útil comparar cada configuración paralela contra la implementación secuencial.

Para ello calculamos:

\[
S_{sec\rightarrow p} = \frac{T_{secuencial}}{T_p}
\]

Si este valor es menor que 1, la versión paralela todavía tarda más que la secuencial.


In [ ]:
promedios["speedup_vs_secuencial"] = (
    tiempo_secuencial / promedios["tiempo_promedio_segundos"]
)

display(promedios)


# 8. Guardar el análisis

El archivo resultante será utilizado por el integrante de gráficas.

Se guardan:

- workers;
- tiempo promedio;
- Speedup;
- eficiencia;
- comparación contra la versión secuencial.


In [ ]:
ruta_salida = RESULTADOS / "analisis.csv"

promedios.to_csv(ruta_salida, index=False)

print("Archivo guardado en:")
print(ruta_salida.resolve())


# 9. Validaciones

Antes de entregar los resultados verificamos que:

- existan exactamente las configuraciones 1, 2 y 4;
- todos los tiempos sean positivos;
- el Speedup de 1 worker sea 1;
- no existan valores vacíos.


In [ ]:
assert set(promedios["workers"]) == {1, 2, 4}
assert (promedios["tiempo_promedio_segundos"] > 0).all()
assert np.isclose(
    promedios.loc[promedios["workers"] == 1, "speedup"].iloc[0],
    1.0
)
assert not promedios.isnull().any().any()

print("Validaciones correctas.")


# 10. Resultados obtenidos

Con los datos registrados en las etapas anteriores se observa que:

- dentro de la versión paralela, aumentar de 1 a 2 y 4 workers reduce el tiempo promedio;
- 4 workers obtiene el menor tiempo entre las configuraciones paralelas;
- el Speedup aumenta al incrementar el número de workers;
- la eficiencia disminuye al agregar workers, por lo que el rendimiento no escala de forma perfectamente proporcional;
- en estas mediciones, la implementación secuencial todavía es más rápida que las configuraciones paralelas.

Esto **no invalida el experimento**. Precisamente uno de los objetivos de la práctica es comprobar mediante mediciones si la paralelización resulta beneficiosa para el problema y el hardware utilizado.


In [ ]:
tabla = promedios.copy()

tabla["tiempo_promedio_segundos"] = tabla["tiempo_promedio_segundos"].round(6)
tabla["speedup"] = tabla["speedup"].round(4)
tabla["eficiencia"] = tabla["eficiencia"].round(4)
tabla["speedup_vs_secuencial"] = tabla["speedup_vs_secuencial"].round(4)

print(f"Tiempo promedio secuencial: {tiempo_secuencial:.6f} s")
display(tabla)


# 11. Entrega al siguiente integrante

Esta rama deja preparado:

```text
resultados/analisis.csv
```

El integrante encargado de las gráficas puede utilizar ese archivo para generar:

- workers vs. tiempo;
- workers vs. Speedup;
- workers vs. eficiencia.

Así cada parte del equipo tiene una responsabilidad separada y verificable en GitHub.
